# Stage 5 §H Phase B — TSP-20 K=10 mix(λ=0.5) step50 100-iter training (Colab T4)

Mix-leaf-eval training run mirroring the Modal entrypoint `run_tsp20_k10_mix_step50` (default λ=0.5). Recipe is §B.4 `tsp20_k10_lv0_step50` verbatim except `leaf_eval: rollout→mix`, `lambda_v: 0.0→1.0`, plus `--mix_lambda 0.5`. From-scratch (no warm-start). 100 iter total, lr step 5e-4→1e-4 at iter 50.

**Apples-to-apples baseline:** §B.4 reached val=3.8576 at iter 100 / 33 min on A10. Phase A (§H.3) on F.6.1.6 showed pure rollout (λ=0) beats mix at inference by ~0.004 — Phase B's hypothesis is that a mix-trained value head will be less biased and therefore the curve flips. Null result (tie or worse than §B.4) is informative — closes the AGFan/Lee question for TSP-20.

**Wall expectation on T4:** §B.4 was ~20 s/iter on A10. T4 is ~2-3× slower at training-shape workloads, so estimate ~60-90 min total for 100 iter. Comfortable inside Colab Pro's 12h session.

**Disconnect-safety:** all outputs (checkpoints + iterations.csv) write directly to Drive at `MyDrive/AM_AlphaGoZero/outputs/tsp_20/<run_name>/`, so a session disconnect doesn't lose work. The run is resumable by re-running cells 2.1 onward (timestamp embedded in run_name guarantees a fresh dir; for true resume from checkpoint we'd add `--load_path`, deferred until/unless we need it).

**Before running:**
1. Add your W&B API key to Colab Secrets as `WANDB_API_KEY` (sidebar → key icon → +). Required for `wandb_mode=online`. If you skip this, set `WANDB_MODE = 'offline'` in §2.1 — local CSV logging still works.
2. (Nothing else — Phase B is from-scratch, no checkpoint to upload.)

## Section 1 — setup (Drive mount + repo + build + smoke)

### 1.1 GPU / Python / CUDA sanity

In [ ]:
import sys, platform
print('python    =', sys.version.split()[0], platform.platform())
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch
print('torch     =', torch.__version__, '  cuda available =', torch.cuda.is_available())
print('cuda      =', torch.version.cuda, '  device =', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

### 1.2 Mount Drive + workspace paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
WORKSPACE = '/content/drive/MyDrive/AM_AlphaGoZero'
REPO_DIR = os.path.join(WORKSPACE, 'repo')
CKPT_DIR = os.path.join(WORKSPACE, 'checkpoints')
OUTPUT_DIR = os.path.join(WORKSPACE, 'outputs')

os.makedirs(WORKSPACE, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('WORKSPACE =', WORKSPACE)
print('REPO_DIR  =', REPO_DIR)
print('OUTPUT_DIR=', OUTPUT_DIR)

### 1.3 Clone or update repo

`git pull --ff-only` will refuse to merge if there are local changes — that's by design. If it fails, the fix is `git -C {REPO_DIR} fetch origin && git -C {REPO_DIR} reset --hard origin/main` (discards any local Drive-side edits, which there shouldn't be).

In [ ]:
REPO_URL = 'https://github.com/LejunZhou/AM_ALPHAGOZERO.git'

if not os.path.isdir(os.path.join(REPO_DIR, '.git')):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print('Repo already cloned; pulling latest...')
    !git -C {REPO_DIR} pull --ff-only

!git -C {REPO_DIR} log --oneline -1

### 1.4 Install package + build C++ MCTS extension

In [ ]:
%cd {REPO_DIR}
!pip install --quiet pybind11
!pip install --quiet --no-deps -e .
!pip install --quiet matplotlib numpy scipy tqdm 'wandb>=0.18.0'

In [ ]:
import sys
sys.path.insert(0, os.path.join(REPO_DIR, 'src'))

# Drop any stale cached imports from prior runs in the same kernel.
for name in [n for n in list(sys.modules) if n.startswith('am_baseline')]:
    del sys.modules[name]

from am_baseline.search.mcts_cpp import _mcts_cpp as _ext  # noqa: F401
from am_baseline.search.mcts import MCTSConfig, MCTSSolver
assert 'mix' in MCTSSolver.VALID_LEAF_EVAL, 'mix mode missing — pull latest commits and rebuild'
print('OK — C++ MCTS extension imports; mix mode registered')
print('VALID_LEAF_EVAL =', MCTSSolver.VALID_LEAF_EVAL)

### 1.5 W&B login (Colab Secrets → WANDB_API_KEY)

If this cell fails or prints `Skipping W&B login`, set `WANDB_MODE = 'offline'` in §2.1. Training will still run; metrics just don't sync to W&B.

In [ ]:
WANDB_OK = False
try:
    from google.colab import userdata
    key = userdata.get('WANDB_API_KEY')
    import wandb
    wandb.login(key=key)
    WANDB_OK = True
    print('W&B login OK')
except Exception as e:
    print('Skipping W&B login:', type(e).__name__, str(e)[:120])
    print('(Will fall back to offline mode in §2.1.)')

### 1.6 Mix smoke test (M1/M2/M3)

Confirms mix mode is wired correctly before launching the 90-minute training run. All three tests must print `|Δ|=0`.

In [ ]:
!cd {REPO_DIR} && PYTHONPATH=src python src/scripts/smoke_mix.py

## Section 2 — Phase B training run

### 2.1 Set up run name + output dir on Drive

Run name mirrors the Modal naming: `tsp20_k10_mix0p5_step50_100iter_<UTC timestamp>`. Outputs land in `OUTPUT_DIR/tsp_20/<run_name>/` so a session disconnect doesn't lose checkpoints or iterations.csv.

In [ ]:
from datetime import datetime, timezone

MIX_LAMBDA = '0.5'   # the canonical AGFan/Lee value; matches `run_tsp20_k10_mix_step50` default
WANDB_MODE = 'online' if WANDB_OK else 'offline'

lam_tag = MIX_LAMBDA.replace('.', 'p')
timestamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')
RUN_NAME = f'tsp20_k10_mix{lam_tag}_step50_100iter_{timestamp}'
RUN_DIR = os.path.join(OUTPUT_DIR, 'tsp_20', RUN_NAME)

print(f'MIX_LAMBDA = {MIX_LAMBDA}')
print(f'WANDB_MODE = {WANDB_MODE}  (set WANDB_MODE = "offline" manually if §1.5 failed)')
print(f'RUN_NAME   = {RUN_NAME}')
print(f'RUN_DIR    = {RUN_DIR}')
print(f'OUTPUT_DIR = {OUTPUT_DIR}  (--output_dir; train_alphazero.py adds tsp_20/<run_name>/)')

### 2.2 Launch training (~60-90 min on T4)

Args lifted from `_f61_args` (Modal entrypoint `run_tsp20_k10_mix_step50`):

- `--graph_size 20 --n_iterations 100 --M_instances 1000`
- `--n_simulations_train 10` (K=10 self-play; §B.4 winner sim count)
- `--train_steps_per_iter 200 --buffer_capacity 5000 --batch_size 512`
- `--gate_every 1 --gate_mode ttest` (paired-t accept per iter, Stage 1 convention)
- `--temperature_schedule step10 --dirichlet_epsilon 0.25 --dirichlet_alpha_factor 10.0`
- `--val_size 10000 --val_seed 42` (greedy val on 10k instances every iter)
- `--leaf_eval mix --mix_lambda 0.5 --lambda_v 1.0` (mix Q-values, vh-trained)
- `--lr_model 5e-4 --lr_decay 0.2 --lr_decay_step_size 50` (5e-4 → 1e-4 at iter 50)
- `--weight_decay 0.0 --max_grad_norm 1.0 --value_target_norm none`
- `--mcts_batch_size 1000` (production batch — same as Phase A)

Output streams to the cell; the same content also lands in `RUN_DIR/iterations.csv` for offline analysis.

In [ ]:
!cd {REPO_DIR} && PYTHONPATH=src python src/scripts/train_alphazero.py \
    --graph_size 20 \
    --n_iterations 100 \
    --M_instances 1000 \
    --n_simulations_train 10 \
    --train_steps_per_iter 200 \
    --buffer_capacity 5000 \
    --batch_size 512 \
    --gate_every 1 \
    --gate_mode ttest \
    --temperature_schedule step10 \
    --val_size 10000 \
    --val_seed 42 \
    --leaf_eval mix \
    --mix_lambda {MIX_LAMBDA} \
    --lambda_v 1.0 \
    --max_grad_norm 1.0 \
    --value_target_norm none \
    --lr_model 5e-4 \
    --lr_decay 0.2 \
    --lr_decay_step_size 50 \
    --weight_decay 0.0 \
    --dirichlet_epsilon 0.25 \
    --dirichlet_alpha_factor 10.0 \
    --mcts_batch_size 1000 \
    --wandb_project am-alphagozero \
    --wandb_mode {WANDB_MODE} \
    --output_dir {OUTPUT_DIR} \
    --run_name {RUN_NAME}

### 2.3 Results summary

Loads `iterations.csv` from the run dir, prints best val + iter, gate accept rate, total wall, and the per-iter trajectory tail. The full table is available at `RUN_DIR/iterations.csv` for downstream analysis.

In [ ]:
import csv, glob
import numpy as np

# The train_alphazero.py CLI appends its own timestamp to opts.run_name, so find the actual dir.
candidates = sorted(glob.glob(os.path.join(OUTPUT_DIR, 'tsp_20', f'{RUN_NAME}*')))
if not candidates:
    raise FileNotFoundError(f'No run dir matching {RUN_NAME}* under {OUTPUT_DIR}/tsp_20/')
ACTUAL_RUN_DIR = candidates[-1]
print('Run dir   =', ACTUAL_RUN_DIR)

ITER_CSV = os.path.join(ACTUAL_RUN_DIR, 'iterations.csv')
with open(ITER_CSV, newline='') as f:
    rows = list(csv.DictReader(f))
print(f'iterations.csv rows = {len(rows)}')

vals   = np.array([float(r['val_avg_cost']) for r in rows if r['val_avg_cost'] != ''])
iters  = np.array([int(r['iter']) for r in rows if r['val_avg_cost'] != ''])
# `accepted` is written as '1' / '0' / '' (empty when not gated). `mcts_wall_s` + `train_wall_s` track per-iter wall.
accepts    = np.array([1 if r.get('accepted', '') == '1' else 0 for r in rows])
gated_mask = np.array([r.get('gated', '0') == '1' for r in rows])
mcts_wall  = np.array([float(r.get('mcts_wall_s', 0) or 0) for r in rows])
train_wall = np.array([float(r.get('train_wall_s', 0) or 0) for r in rows])

best_idx = int(np.argmin(vals))
print(f'\n=== Phase B summary — λ={MIX_LAMBDA} ===')
print(f'Best val:    {vals[best_idx]:.5f}  @ iter {iters[best_idx]}')
print(f'Final val:   {vals[-1]:.5f}        @ iter {iters[-1]}')
print(f'§B.4 baseline (lv0 K=10 step50, A10): 3.8576 @ iter 100 / 33 min')
print(f'§B.3 F.6.1.6 ceiling                : 3.8578 @ iter 365')
n_gated = int(gated_mask.sum())
if n_gated > 0:
    print(f'Gate accept rate: {accepts.sum()}/{n_gated} gated iters = {accepts.sum()/n_gated:.1%}')
print(f'Total wall:  {mcts_wall.sum()/60:.1f} min MCTS + {train_wall.sum()/60:.1f} min train = {(mcts_wall.sum()+train_wall.sum())/60:.1f} min')

# Tail of the trajectory
print(f'\nLast 10 iters:')
print(f'{"iter":>5} {"val":>9} {"gated":>6} {"accepted":>9} {"mcts_s":>8} {"train_s":>8}')
for r in rows[-10:]:
    print(f'{int(r["iter"]):>5} {float(r["val_avg_cost"]):>9.5f} {r.get("gated","?"):>6} {r.get("accepted","?"):>9} {float(r.get("mcts_wall_s",0) or 0):>8.1f} {float(r.get("train_wall_s",0) or 0):>8.1f}')

print(f'\nFull table:    {ITER_CSV}')
print(f'Checkpoints:   ls {ACTUAL_RUN_DIR}/iter-*_accepted.pt')

### 2.4 Next steps

Paste the §2.3 summary back to the main thread. I'll:

- Record into `_progress/stage5_mix_leafeval_progress.md` §H.4 with the apples-to-apples verdict vs §B.4 (3.8576).
- Pull the best-iter checkpoint locally from Drive at `MyDrive/AM_AlphaGoZero/outputs/tsp_20/<RUN_NAME>/iter-<best>_accepted.pt`.
- Queue Phase C — 1000-instance Gurobi-anchored eval via `eval_tsp20_full_comparison.py` on the mix checkpoint, F.6.1.6 best, and lv0 iter-199.

If the run disconnected mid-way, the partial `iterations.csv` and any accepted checkpoints up to that point are persisted in `RUN_DIR` on Drive — we can still extract a partial trajectory or restart from the last accepted checkpoint (would need a small `--load_path` patch to the entrypoint).